## LOBO Calibration on Held-out HC

Same checks as `1_cv_analysis.ipynb`, applied to HC of a batch the engine never saw. Every held-out HC
is compared with its own CV score (same sample, batch seen in training), so the gap between the two
columns is what an unseen batch costs. Detection performance on injected signals lives in
`5_in_silico_method_comparison.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import json
import os
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde, kurtosis, norm, skew

os.environ["NM_PIPELINE"] = "p3"
sys.path.insert(0, str(Path.cwd().parents[1]))
import MixedEffectsModeling.config as config
from MixedEffectsModeling.core.calibration import bh_fdr_reject
from MixedEffectsModeling.validation.lobo_engine import selected_batches
from viz_style import apply_style, FS_ANNOT_LG, FS_ANNOT_SM
apply_style()

LOBO = config.LOBO_MIXED_DIR
CV = config.CV_MIXED_DIR
FIG = LOBO / 'Figures'
FIG.mkdir(parents=True, exist_ok=True)

TARGET = {'mean': 0.0, 'std': 1.0, 'skew': 0.0, 'kurt': 0.0, 'exceed': 0.05}
Q_GENE = (0.01, 0.005)
Q_SAMPLE = 0.05
COL = {'CV (seen batch)': '#2b5c8f', 'LOBO (unseen batch)': '#d62828'}

### 0. Held-out HC and gene set

HC of each held-out batch that pass the OOD filter (`ood_mask.npy`, written by `lobo_engine.py`).
CV Z are read for the same samples from `cv_zscores*.pkl`, whose arrays are the fold test sets concatenated in fold
order (`cv_folds.json`); genes that failed in any fold are dropped.

In [ ]:
folds = json.load(open(config.OUTRIDER_COMPARISON_DIR / 'cv_folds.json'))
cv_order = np.concatenate([folds[k]['test'] for k in sorted(folds, key=int)])
cv_pos = pd.Series(np.arange(len(cv_order)), index=cv_order)
cv_raw = {g: z for g, z in pickle.load(open(CV / 'cv_zscores.pkl', 'rb')).items() if len(z) == len(cv_order)}
cv_sh = pickle.load(open(CV / 'cv_zscores_shash.pkl', 'rb'))

metas = selected_batches()
lobo_genes = [pickle.load(open(LOBO / m['batch_dir'] / 'gene_names.pkl', 'rb')) for m in metas]
GENES = sorted(set.intersection(*map(set, lobo_genes)) & set(cv_raw))

parts, rows = [], []
for m, g in zip(metas, lobo_genes):
    bdir = LOBO / m['batch_dir']
    keep = np.array(m['test_is_hc']) & np.load(bdir / 'ood_mask.npy')
    names = np.array(m['test_names'])[keep]
    gi = pd.Index(g).get_indexer(GENES)
    parts.append(dict(batch=m['batch_id'], names=names,
                      lobo_raw=np.load(bdir / 'Z_test.npy')[keep][:, gi].astype(float),
                      lobo_sh=np.load(bdir / 'Z_test_shash.npy')[keep][:, gi].astype(float)))
    rows.append(dict(batch=m['batch_id'], n_hc_test=int(np.sum(m['test_is_hc'])), n_hc_ood_kept=int(keep.sum())))

pos = cv_pos.loc[np.concatenate([p['names'] for p in parts])].values
BATCH = np.concatenate([np.repeat(p['batch'], len(p['names'])) for p in parts])
Z = {('CV (seen batch)', 'raw'): np.column_stack([cv_raw[g][pos] for g in GENES]).astype(float),
     ('CV (seen batch)', 'shash'): np.column_stack([cv_sh[g][pos] for g in GENES]).astype(float),
     ('LOBO (unseen batch)', 'raw'): np.vstack([p['lobo_raw'] for p in parts]),
     ('LOBO (unseen batch)', 'shash'): np.vstack([p['lobo_sh'] for p in parts])}
for k in Z:
    Z[k][~np.isfinite(Z[k])] = np.nan

print(f'genes shared by CV and all {len(metas)} LOBO fits: {len(GENES)}')
pd.DataFrame(rows)

### A1. Per-gene Z moments (before / after SHASH)

Each gene's mean, SD, skewness and excess kurtosis over the pooled held-out HC. Dashed line = ideal N(0, 1).

In [ ]:
def moments(z):
    return pd.DataFrame({'mean': np.nanmean(z, 0), 'std': np.nanstd(z, 0),
                         'skew': skew(z, 0, nan_policy='omit'), 'kurt': kurtosis(z, 0, nan_policy='omit')},
                        index=GENES)


MOM = {k: moments(v) for k, v in Z.items()}

fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for col, (stat, lab) in enumerate([('mean', 'Mean'), ('std', 'SD'), ('skew', 'Skewness'), ('kurt', 'Excess kurtosis')]):
    lo, hi = np.nanpercentile(np.concatenate([MOM[k][stat].values for k in MOM]), [0.5, 99.5])
    grid = np.linspace(lo, hi, 512)
    for row, kind in enumerate(['raw', 'shash']):
        ax = axes[row, col]
        for arm, c in COL.items():
            v = MOM[(arm, kind)][stat].dropna().values
            d = gaussian_kde(v)(grid)
            ax.fill_between(grid, d, color=c, alpha=0.25, lw=0)
            ax.plot(grid, d, color=c, lw=1.2, label=arm.split(' ')[0])
        ax.axvline(TARGET[stat], color='black', ls='--', lw=1.2)
        ax.set_xlim(lo, hi); ax.set_ylim(bottom=0)
        ax.set_title(f"{lab}, {'before' if kind == 'raw' else 'after'} SHASH", fontsize=FS_ANNOT_LG)
        ax.set_ylabel('Density' if col == 0 else '')
        ax.tick_params(labelsize=FS_ANNOT_SM)
        if (row, col) == (0, 0):
            ax.legend(frameon=False, fontsize=FS_ANNOT_SM, loc='upper left')
        ax.grid(True, ls=':', alpha=0.4)
plt.tight_layout()
plt.savefig(FIG / 'lobo_moments_cv_vs_lobo.png', dpi=300, bbox_inches='tight')
plt.show()

pd.DataFrame({f'{arm} / {kind}': MOM[(arm, kind)].median() for arm, kind in MOM}).T.round(4)

### A2. Exceedance calibration

Left: fraction of HC x gene cells with two-sided p < alpha against alpha (diagonal = calibrated), SHASH Z,
one line per held-out batch. Right: per-gene |Z| > 1.96 rate per batch (target 0.05).

In [ ]:
ALPHAS = np.logspace(-4, np.log10(0.5), 25)
batches = list(dict.fromkeys(BATCH))


def exceed_curve(z):
    p = 2 * norm.sf(np.abs(z[np.isfinite(z)]))
    return np.array([(p < a).mean() for a in ALPHAS])


fig, (ax, bx) = plt.subplots(1, 2, figsize=(9, 4), gridspec_kw={'width_ratios': [1, 1.4]})
for arm, c in COL.items():
    for b in batches:
        ax.plot(ALPHAS, exceed_curve(Z[(arm, 'shash')][BATCH == b]), color=c, lw=1, alpha=0.7)
    ax.plot([], [], color=c, label=arm)
ax.plot(ALPHAS, ALPHAS, color='black', ls='--', lw=1)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('nominal alpha'); ax.set_ylabel('observed exceedance')
ax.set_title('Exceedance vs nominal (SHASH)', fontsize=FS_ANNOT_LG)
ax.legend(frameon=False, fontsize=FS_ANNOT_SM)

rate = {arm: [100 * np.nanmean(np.abs(Z[(arm, 'shash')][BATCH == b]) > 1.96, 0) for b in batches] for arm in COL}
x = np.arange(len(batches)); w = 0.38
for k, (arm, c) in enumerate(COL.items()):
    bp = bx.boxplot(rate[arm], positions=x + (k - 0.5) * w, widths=w * 0.9, showfliers=False, patch_artist=True,
                    medianprops=dict(color='black', lw=2.2))
    for b_ in bp['boxes']:
        b_.set_facecolor(c); b_.set_alpha(0.5)
    bx.plot([], [], color=c, lw=6, alpha=0.5, label=arm)
bx.axhline(100 * TARGET['exceed'], color='black', ls='--', lw=1)
bx.set_ylim(0, 50)
bx.set_xticks(x); bx.set_xticklabels([b.replace(' et al._Batch_', '_') for b in batches], rotation=45, ha='right')
bx.set_ylabel('% of samples with |Z| > 1.96 per gene'); bx.set_title('Per-gene exceedance by batch', fontsize=FS_ANNOT_LG)
bx.legend(frameon=False, fontsize=FS_ANNOT_SM)
plt.tight_layout()
plt.savefig(FIG / 'lobo_exceedance_calibration.png', dpi=300, bbox_inches='tight')
plt.show()

### A3. BH outliers

Left: per gene, BH across the pooled held-out HC (as in `1_cv_analysis.ipynb`), number of outlier
samples, SHASH Z. Right two panels: per sample, BH across genes (q = 0.05), number of significant genes as a count and
as a percentage of all tested genes.

In [ ]:
CAT_LABELS = ['0 (Best)', '1', '2', '3', '4', '5+']


def gene_outliers(z, q):
    out = np.zeros(z.shape[1], int)
    for j in range(z.shape[1]):
        v = z[:, j][np.isfinite(z[:, j])]
        out[j] = bh_fdr_reject(2 * norm.sf(np.abs(v)), q=q).sum() if len(v) else 0
    return out


def sample_nsig(z, q):
    return np.array([bh_fdr_reject(2 * norm.sf(np.abs(r[np.isfinite(r)])), q=q).sum() for r in z])


NOUT = {(arm, q): gene_outliers(Z[(arm, 'shash')], q) for arm in COL for q in Q_GENE}
NSIG = {arm: sample_nsig(Z[(arm, 'shash')], Q_SAMPLE) for arm in COL}

fig, axes = plt.subplots(1, len(Q_GENE) + 2, figsize=(17, 3.8), gridspec_kw={'width_ratios': [1, 1, 1.3, 1.3]})
x = np.arange(len(CAT_LABELS)); w = 0.38
for ax, q in zip(axes, Q_GENE):
    for k, (arm, c) in enumerate(COL.items()):
        n = pd.cut(np.clip(NOUT[(arm, q)], 0, 5), bins=[-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5], labels=CAT_LABELS)
        vals = pd.Series(n).value_counts().reindex(CAT_LABELS).fillna(0).values
        bars = ax.bar(x + (k - 0.5) * w, vals, w, color=c, edgecolor='black', label=arm)
        for b_ in bars:
            if b_.get_height() > 0:
                ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f'{int(b_.get_height())}',
                        ha='center', va='bottom', fontsize=FS_ANNOT_SM)
    ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(CAT_LABELS)
    ax.set_xlabel('outlier samples per gene'); ax.set_ylabel('# genes (log)')
    ax.set_title(f'BH q={q:.3f}: outliers per gene', fontsize=FS_ANNOT_LG)
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM)

def nsig_box(ax, scale):
    x = np.arange(len(batches))
    for k, (arm, c) in enumerate(COL.items()):
        data = [NSIG[arm][BATCH == b] * scale for b in batches]
        bp = ax.boxplot(data, positions=x + (k - 0.5) * w, widths=w * 0.9, showfliers=False, patch_artist=True,
                    medianprops=dict(color='black', lw=2.2))
        for b_ in bp['boxes']:
            b_.set_facecolor(c); b_.set_alpha(0.5)
        for i, d in enumerate(data):
            ax.scatter(np.full(len(d), x[i] + (k - 0.5) * w), d, s=6, color=c, alpha=0.6, zorder=3)
        ax.plot([], [], color=c, lw=6, alpha=0.5, label=arm)
    ax.set_xticks(x); ax.set_xticklabels([b.replace(' et al._Batch_', '_') for b in batches], rotation=45, ha='right')
    ax.legend(frameon=False, fontsize=FS_ANNOT_SM)


ax = axes[-2]
nsig_box(ax, 1)
ax.set_yscale('symlog', linthresh=1)
ax.set_ylim(bottom=0)
ax.set_ylabel(f'BH-significant genes (q={Q_SAMPLE})')
ax.set_title('Per-sample false calls (count)', fontsize=FS_ANNOT_LG)

ax = axes[-1]
nsig_box(ax, 100 / len(GENES))
ax.set_ylim(0, 100)
ax.set_ylabel(f'% of {len(GENES):,} tested genes')
ax.set_title('Per-sample false calls (%)', fontsize=FS_ANNOT_LG)
plt.tight_layout()
plt.savefig(FIG / 'lobo_bh_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

### Summary per held-out batch

In [ ]:
summ = []
for b in batches:
    r = BATCH == b
    for arm in COL:
        z = Z[(arm, 'shash')][r]
        summ.append(dict(batch=b, arm=arm, n_hc=int(r.sum()),
                         median_gene_mean=np.nanmedian(np.nanmean(z, 0)), median_gene_sd=np.nanmedian(np.nanstd(z, 0)),
                         exceed_196=np.nanmean(np.abs(z) > 1.96), exceed_329=np.nanmean(np.abs(z) > 3.29),
                         nsig_median=np.median(NSIG[arm][r]), nsig_mean=NSIG[arm][r].mean(),
                         nsig_pct_median=100 * np.median(NSIG[arm][r]) / len(GENES),
                         nsig_pct_mean=100 * NSIG[arm][r].mean() / len(GENES),
                         frac_samples_nsig0=(NSIG[arm][r] == 0).mean()))
summ = pd.DataFrame(summ)
summ.to_csv(LOBO / 'lobo_hc_calibration_summary.csv', index=False)
summ.pivot(index='batch', columns='arm').round(4)